In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------------ 상수 (Protocol 11절)
CODES = [142, 215, 216, 218, 219, 235, 238, 249, 322, 325, 394, 421, 429,
         612, 629, 632, 634, 728]
N = 72
MIN_SIZE, C, K = 6, 1.0, 2          # trend cost (k=2)
W = 12
MIN_SIDE = 6
Z_SELECT = 2.0
HAC_LAGS = 2
Q_SUP, Q_BORD = 0.05, 0.10
LEVEL_PCT, TREND_PP = 10.0, 5.0
PERSIST_MIN_GAP = 0.05
METRICS = ["V", "Q", "UV"]

DATA = "meft_2020-2025.csv"
OUT = "output"
os.makedirs(OUT, exist_ok=True)
MONTHS = pd.period_range("2020-01", periods=N, freq="M").strftime("%Y-%m").tolist()


def save(df, name):
    df.to_csv(os.path.join(OUT, name), index=False, encoding="utf-8-sig")
    print(f"  saved {name} ({len(df)} rows)")


# ------------------------------------------------------------------ S0 로드·QA
print("[S0] 로드·QA")
raw = pd.read_csv(DATA)
raw = raw[raw["약효분류코드"].isin(CODES)]
qa_rows, panel = [], {}
for code in CODES:
    d = raw[raw["약효분류코드"] == code]
    n_rows = len(d)
    n_dup = int(d["진료년월"].duplicated().sum())
    present = set(d["진료년월"]) & set(MONTHS)
    n_miss_month = N - len(present)
    reasons = []
    if n_dup:
        reasons.append("DUPLICATE_MONTH")
    if n_miss_month:
        reasons.append("MISSING_MONTH")
    V = Q = None
    if not n_dup:
        d = d.set_index("진료년월").reindex(MONTHS)
        V = d["사용금액_백만원"].astype(float).to_numpy()
        Q = d["수량_천개"].astype(float).to_numpy()
        nan_v, nan_q = int(np.isnan(V).sum()), int(np.isnan(Q).sum())
        np_v, np_q = int((V <= 0).sum()), int((Q <= 0).sum())
        if nan_v or nan_q:
            reasons.append("MISSING_VALUE")
        if np_v or np_q:
            reasons.append("NONPOSITIVE_VALUE")
    else:
        nan_v = nan_q = np_v = np_q = np.nan
    included = not reasons
    qa_rows.append(dict(code=code, n_rows=n_rows, n_dup=n_dup, n_missing_month=n_miss_month,
                        n_nan_V=nan_v, n_nan_Q=nan_q, n_nonpos_V=np_v, n_nonpos_Q=np_q,
                        included=included, reason=";".join(reasons) if reasons else "OK"))
    if included:
        panel[code] = dict(V=V, Q=Q)
qa = pd.DataFrame(qa_rows)
save(qa, "qa_summary.csv")
inc = list(panel)
print(f"  포함 {len(inc)}개 / 제외 {len(CODES) - len(inc)}개")
if not inc:
    sys.exit("중단: 포함 코드 0개")


# ------------------------------------------------------------------ S1 로그·계절조정
def seasonal_adjust(y):
    w = np.r_[0.5, np.ones(11), 0.5] / 12.0
    ma = np.full(N, np.nan)
    for t in range(6, N - 6):
        ma[t] = w @ y[t - 6:t + 7]
    rs = y - ma
    idx = np.array([np.nanmedian(rs[m::12]) for m in range(12)])
    idx = idx - idx.mean()
    return y - idx[np.arange(N) % 12]


print("[S1] 로그·계절조정")
sa, sa_rows = {}, []
for code in inc[:]:
    try:
        v = seasonal_adjust(np.log(panel[code]["V"]))
        q = seasonal_adjust(np.log(panel[code]["Q"]))
        if not (np.isfinite(v).all() and np.isfinite(q).all()):
            raise ValueError("non-finite")
    except Exception:
        qa.loc[qa.code == code, ["included", "reason"]] = [False, "SEASONAL_FAILED"]
        inc.remove(code)
        continue
    sa[code] = {"V": v, "Q": q, "UV": v - q}      # logUV_sa = logV_sa - logQ_sa
    for t in range(N):
        sa_rows.append(dict(code=code, t=t, ym=MONTHS[t], logV_sa=v[t], logQ_sa=q[t],
                            logUV_sa=v[t] - q[t]))
save(pd.DataFrame(sa_rows), "series_sa.csv")
save(qa, "qa_summary.csv")


# ------------------------------------------------------------------ S2 변화점 탐지
def detect_breaks(y_std):
    n = len(y_std)
    t = np.arange(n, dtype=float)
    cs = lambda a: np.r_[0.0, np.cumsum(a)]
    St, Stt, Sy, Sty, Syy = cs(t), cs(t * t), cs(y_std), cs(t * y_std), cs(y_std ** 2)

    def cost(i, j):
        m = j - i
        st, stt = St[j] - St[i], Stt[j] - Stt[i]
        sy, sty, syy = Sy[j] - Sy[i], Sty[j] - Sty[i], Syy[j] - Syy[i]
        sxx = stt - st ** 2 / m
        sxy = sty - st * sy / m
        return max(syy - sy ** 2 / m - sxy ** 2 / sxx, 0.0)

    pen = C * K * np.log(n)
    F = np.full(n + 1, np.inf)
    F[0] = -pen
    prev = np.zeros(n + 1, dtype=int)
    for j in range(MIN_SIZE, n + 1):
        best, bi = np.inf, -1
        for i in range(0, j - MIN_SIZE + 1):
            if not np.isfinite(F[i]):
                continue
            val = F[i] + cost(i, j) + pen
            if val < best - 1e-12:           # 동률이면 가장 이른 i
                best, bi = val, i
        F[j], prev[j] = best, bi
    brk, j = [], n
    while j > 0:
        i = prev[j]
        if i > 0:
            brk.append(i)
        j = i
    return sorted(brk)


print("[S2] 변화점 탐지 (V, Q)")
m1_rows = []
for code in inc:
    for m in ["V", "Q"]:
        y = sa[code][m]
        d = np.diff(y)
        sigma = 1.4826 * np.median(np.abs(d - np.median(d))) / np.sqrt(2)
        if not np.isfinite(sigma) or sigma == 0:
            m1_rows.append(dict(code=code, metric=m, b=np.nan, break_date="", sigma_hat=sigma,
                                status="M1_STANDARDIZATION_FAILED"))
            continue
        brk = detect_breaks((y - np.median(y)) / sigma)
        if not brk:
            m1_rows.append(dict(code=code, metric=m, b=np.nan, break_date="", sigma_hat=sigma,
                                status="NO_BREAK_DETECTED"))
        for b in brk:
            m1_rows.append(dict(code=code, metric=m, b=b, break_date=MONTHS[b], sigma_hat=sigma,
                                status="DETECTED"))
m1 = pd.DataFrame(m1_rows)
save(m1, "m1_candidates.csv")
print("  탐지 후보:", m1[m1.status == "DETECTED"].groupby("metric").size().to_dict())


# ------------------------------------------------------------------ ITS 공통
def window(b):
    return max(0, b - W), min(N - 1, b + W - 1)


def its_design(b):
    lo, hi = window(b)
    t = np.arange(lo, hi + 1)
    x = t - (b - 0.5)
    post = (t >= b).astype(float)
    return t, np.column_stack([np.ones_like(x), x, post, x * post]), lo, hi


# ------------------------------------------------------------------ S3 선정
print("[S3] 선정")
m2_rows = []
for r in m1[m1.status == "DETECTED"].itertuples():
    b = int(r.b)
    t, X, lo, hi = its_design(b)
    n_pre, n_post = b - lo, hi - b + 1
    row = dict(code=r.code, metric=r.metric, b=b, date=MONTHS[b], n_pre=n_pre, n_post=n_post,
               z_level=np.nan, z_trend=np.nan)
    if n_pre < MIN_SIDE or n_post < MIN_SIDE:
        row.update(status="HELD_CENSORED",
                   reason="START_OF_DATA" if n_pre < MIN_SIDE else "END_OF_DATA")
    else:
        res = sm.OLS(sa[r.code][r.metric][lo:hi + 1], X).fit()
        zl, zt = res.params[2] / res.bse[2], res.params[3] / res.bse[3]
        row.update(z_level=zl, z_trend=zt)
        if abs(zl) >= Z_SELECT or abs(zt) >= Z_SELECT:
            row.update(status="SELECTED", reason="")
        else:
            row.update(status="REJECTED", reason="LOW_SNR")
    m2_rows.append(row)
m2 = pd.DataFrame(m2_rows)
save(m2, "m2_breaks.csv")
print("  ", m2.groupby(["metric", "status"]).size().to_dict())

sel = m2[m2.status == "SELECTED"]
bk = (sel.groupby(["code", "b"])["metric"]
      .apply(lambda s: "+".join(sorted(s, key=["V", "Q"].index))).reset_index()
      .rename(columns={"metric": "source_metrics"}).sort_values(["code", "b"]).reset_index(drop=True))
bk["break_id"] = [f"B{i + 1:03d}" for i in range(len(bk))]
bk["date"] = bk["b"].map(lambda b: MONTHS[b])
print(f"  회귀 대상 break_id {len(bk)}개")

# ------------------------------------------------------------------ S4/S5 효과크기·검증, S6 지속성
print("[S4~S6] 효과크기·검증·지속성")
eff_rows, val_rows, per_rows = [], [], []
for r in bk.itertuples():
    b = int(r.b)
    t, X, lo, hi = its_design(b)
    n = len(t)
    n_pre, n_post = b - lo, hi - b + 1
    if n_pre < MIN_SIDE or n_post < MIN_SIDE:
        continue
    tcrit = stats.t.ppf(0.975, n - 4)
    for m in METRICS:
        y = sa[r.code][m]
        base = dict(break_id=r.break_id, code=r.code, date=r.date, metric=m)
        try:
            res = sm.OLS(y[lo:hi + 1], X).fit(cov_type="HAC",
                                              cov_kwds={"maxlags": HAC_LAGS, "use_correction": True},
                                              use_t=True)
            b0, b1, b2, b3 = res.params
            se2, se3 = res.bse[2], res.bse[3]
            ols = sm.OLS(y[lo:hi + 1], X).fit()
            if not np.isfinite([b0, b1, b2, b3]).all():
                raise ValueError("non-finite")
        except Exception:
            eff_rows.append(dict(base, status="REGRESSION_FAILED"))
            continue
        g = lambda z: 100 * (np.exp(z) - 1)
        pre_g, post_g = g(12 * b1), g(12 * (b1 + b3))
        eff_rows.append(dict(
            base, n=n, n_pre=n_pre, n_post=n_post, b0=b0, b1=b1, b2=b2, b3=b3,
            level_shift_pct=g(b2), level_ci_lo=g(b2 - tcrit * se2), level_ci_hi=g(b2 + tcrit * se2),
            trend_shift_log_ann=12 * b3, trend_ci_lo=12 * (b3 - tcrit * se3),
            trend_ci_hi=12 * (b3 + tcrit * se3),
            pre_growth_ann_pct=pre_g, post_growth_ann_pct=post_g, trend_shift_pp=post_g - pre_g,
            status="OK"))
        for eff, idx in [("level", 2), ("trend", 3)]:
            p = res.pvalues[idx]
            val_rows.append(dict(break_id=r.break_id, metric=m, effect=eff,
                                 p_hac2=p if np.isfinite(p) else np.nan, p_ols=ols.pvalues[idx]))
        # S6 지속성 (Level만): pre 회귀선 외삽 대비 gap
        tp = np.arange(b, hi + 1)[:12]
        gap = y[tp] - (b0 + b1 * (tp - (b - 0.5)))
        nb = len(gap) // 3
        blocks = [gap[3 * k:3 * k + 3].mean() for k in range(nb)]
        init = blocks[0]
        ratio = blocks[-1] / init if (nb >= 2 and init != 0) else np.nan
        if abs(init) < PERSIST_MIN_GAP:
            label = "INDETERMINATE"
        elif nb < 2:
            label = "CENSORED"
        elif ratio < 0:
            label = "REVERSED"
        elif ratio < 0.25:
            label = "TEMPORARY"
        elif ratio < 0.75:
            label = "PARTIAL_DECAY"
        elif ratio <= 1.25:
            label = "PERSISTENT"
        else:
            label = "AMPLIFIED"
        per_rows.append(dict(base, initial_gap=init, ratio=ratio, label=label, last_block=nb,
                             observed_months=len(gap), trend_label="NOT_RUN"))
eff = pd.DataFrame(eff_rows)
val = pd.DataFrame(val_rows)
per = pd.DataFrame(per_rows)
ok = val.p_hac2.notna()
val["q"] = np.nan
val.loc[ok, "q"] = multipletests(val.loc[ok, "p_hac2"], alpha=Q_SUP, method="fdr_bh")[1]
val["status"] = np.where(val.q < Q_SUP, "SUPPORTED",
                         np.where(val.q < Q_BORD, "BORDERLINE",
                                  np.where(val.q.notna(), "NOT_SUPPORTED", "HAC_FAILED")))
save(eff, "m4_effects.csv")
save(val, "m6_validation.csv")
save(per, "m5_persistence.csv")
print("  FDR family 크기:", int(ok.sum()), "| 상태:", val.status.value_counts().to_dict())

# ------------------------------------------------------------------ S7 분류
print("[S7] 분류")
effi = eff.set_index(["break_id", "metric"])
qi = val.set_index(["break_id", "metric", "effect"])[["q"]]
peri = per.set_index(["break_id", "metric"])


def sign(x):
    return 1 if x > 0 else -1


def metric_status(bid, m):
    if (bid, m) not in effi.index or effi.loc[(bid, m), "status"] != "OK":
        return "REGRESSION_FAILED", ""
    e = effi.loc[(bid, m)]
    ql = qi.loc[(bid, m, "level"), "q"] if (bid, m, "level") in qi.index else np.nan
    qt = qi.loc[(bid, m, "trend"), "q"] if (bid, m, "trend") in qi.index else np.nan
    lm, tm = abs(e.level_shift_pct) >= LEVEL_PCT, abs(e.trend_shift_pp) >= TREND_PP
    ls, ts = ql < Q_SUP, qt < Q_SUP          # NaN은 False
    qual_l, qual_t = lm and ls, tm and ts
    if qual_l or qual_t:
        dl = sign(e.level_shift_pct) if qual_l else None
        dt = sign(e.trend_shift_pp) if qual_t else None
        ctype = "LEVEL_AND_TREND" if (qual_l and qual_t) else ("LEVEL" if qual_l else "TREND")
        if qual_l and qual_t and dl != dt:
            return "STRUCTURAL_MIXED", ctype
        d = dl if qual_l else dt
        return ("STRUCTURAL_UP" if d > 0 else "STRUCTURAL_DOWN"), ctype
    if lm or tm:
        return "EFFECT_NOT_SUPPORTED", ""
    if ls or ts:
        return "SUPPORTED_SMALL_EFFECT", ""
    return "NO_CLEAR_CHANGE", ""


m7_rows = []
for r in bk.itertuples():
    bid = r.break_id
    row = dict(break_id=bid, code=r.code, date=r.date, source_metrics=r.source_metrics)
    st = {}
    for m in METRICS:
        st[m], ct = metric_status(bid, m)
        row[f"{m}_status"], row[f"{m}_change_type"] = st[m], ct
    try:
        ev, eq = effi.loc[(bid, "V")], effi.loc[(bid, "Q")]
        div = bool(sign(ev.level_shift_pct) != sign(eq.level_shift_pct)
                   or sign(ev.trend_shift_pp) != sign(eq.trend_shift_pp))
    except Exception:
        div = False
    uv_sup = bool((qi.loc[(bid, "UV", "level"), "q"] if (bid, "UV", "level") in qi.index else np.nan) < Q_SUP or \
                  (qi.loc[(bid, "UV", "trend"), "q"] if (bid, "UV", "trend") in qi.index else np.nan) < Q_SUP)
    q_struct = st["Q"].startswith("STRUCTURAL")
    v_struct = st["V"].startswith("STRUCTURAL")
    if v_struct:
        if q_struct and uv_sup:
            mech = "VOLUME_AND_UNIT_VALUE"
        elif q_struct:
            mech = "VOLUME_DRIVEN"
        elif uv_sup:
            mech = "UNIT_VALUE_DRIVEN"
        else:
            mech = "UNATTRIBUTED"
    else:
        mech = ""
    if st["V"] == "STRUCTURAL_UP":
        overall = "STRUCTURAL_GROWTH"
    elif st["V"] == "STRUCTURAL_DOWN":
        overall = "STRUCTURAL_DECLINE"
    elif st["V"] == "STRUCTURAL_MIXED":
        overall = "STRUCTURAL_MIXED"
    elif uv_sup:
        overall = "RECONFIGURATION_ONLY"
    elif st["V"] == "EFFECT_NOT_SUPPORTED":
        overall = "CANDIDATE_NOT_SUPPORTED"
    else:
        overall = "NO_CLEAR_CHANGE"
    pl = peri.loc[(bid, "V"), "label"] if (bid, "V") in peri.index else ""
    row.update(divergence_flag=div, structural_reconfiguration=uv_sup, mechanism=mech,
               overall_status=overall, persistence_level_V=pl)
    m7_rows.append(row)
m7 = pd.DataFrame(m7_rows)
save(m7, "m7_classification.csv")
print("  overall:", m7.overall_status.value_counts().to_dict() if len(m7) else {})

cs_rows = []
for code in CODES:
    q_ = qa[qa.code == code].iloc[0]
    row = dict(code=code, included=bool(q_.included), exclusion_reason=q_.reason)
    if code in inc:
        row["n_candidates_V"] = int(((m1.code == code) & (m1.metric == "V") & (m1.status == "DETECTED")).sum())
        row["n_candidates_Q"] = int(((m1.code == code) & (m1.metric == "Q") & (m1.status == "DETECTED")).sum())
        row["n_selected_V"] = int(((m2.code == code) & (m2.metric == "V") & (m2.status == "SELECTED")).sum())
        row["n_selected_Q"] = int(((m2.code == code) & (m2.metric == "Q") & (m2.status == "SELECTED")).sum())
        sub = m7[m7.code == code]
        row["n_breaks"] = len(sub)
        for s in ["STRUCTURAL_GROWTH", "STRUCTURAL_DECLINE", "STRUCTURAL_MIXED", "RECONFIGURATION_ONLY",
                  "CANDIDATE_NOT_SUPPORTED", "NO_CLEAR_CHANGE"]:
            row[s] = int((sub.overall_status == s).sum()) if len(sub) else 0
        row["note"] = "" if len(sub) else "NO_BREAK_SELECTED"
    cs_rows.append(row)
save(pd.DataFrame(cs_rows), "code_summary.csv")

# ------------------------------------------------------------------ S8 Shift-Share

[S0] 로드·QA
  saved qa_summary.csv (18 rows)
  포함 18개 / 제외 0개
[S1] 로그·계절조정
  saved series_sa.csv (1296 rows)
  saved qa_summary.csv (18 rows)
[S2] 변화점 탐지 (V, Q)
  saved m1_candidates.csv (81 rows)
  탐지 후보: {'Q': 31, 'V': 43}
[S3] 선정
  saved m2_breaks.csv (74 rows)
   {('Q', 'REJECTED'): 8, ('Q', 'SELECTED'): 23, ('V', 'REJECTED'): 9, ('V', 'SELECTED'): 34}
  회귀 대상 break_id 50개
[S4~S6] 효과크기·검증·지속성
  saved m4_effects.csv (150 rows)
  saved m6_validation.csv (300 rows)
  saved m5_persistence.csv (150 rows)
  FDR family 크기: 300 | 상태: {'SUPPORTED': 150, 'NOT_SUPPORTED': 130, 'BORDERLINE': 20}
[S7] 분류
  saved m7_classification.csv (50 rows)
  overall: {'STRUCTURAL_GROWTH': 17, 'RECONFIGURATION_ONLY': 12, 'STRUCTURAL_DECLINE': 10, 'STRUCTURAL_MIXED': 5, 'CANDIDATE_NOT_SUPPORTED': 3, 'NO_CLEAR_CHANGE': 3}
  saved code_summary.csv (18 rows)
[S8] Shift-Share (2020 vs 2025, 포함 대상군 내부 시장)
  시장 성장률 g = 0.7158, 항등식 최대오차 = 2.33e-10
  saved m8_shiftshare.csv (18 rows)
완료. 다음: python plot.py output


In [ ]:
# 일반 터미널 환경
# pip install koreanize-matplotlib

# 구글 코랩(Colab) 또는 주피터 노트북 내부에서 실행할 때
!pip install koreanize-matplotlib
# [1] 폰트 설치 (1회만, 런타임 재시작 시 다시)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
print("폰트 설치 완료")
# ============================================================
# 나눔폰트 직접 다운로드 + matplotlib 등록 (Colab 확실 버전)
# ============================================================
import os, matplotlib, matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# 1) matplotlib 캐시 폴더 재생성
cache_dir = matplotlib.get_cachedir()
os.makedirs(cache_dir, exist_ok=True)
print(f"cache dir: {cache_dir}")

# 2) 폰트 저장 폴더
font_dir = "/usr/share/fonts/truetype/nanum"
os.makedirs(font_dir, exist_ok=True)

# 3) wget으로 나눔폰트 직접 다운로드
!wget -q -O {font_dir}/NanumGothic.ttf "https://github.com/naver/nanumfont/raw/master/fonts/NanumFontSetup_TTF_GOTHIC/NanumGothic.ttf"
!wget -q -O {font_dir}/NanumBarunGothic.ttf "https://github.com/naver/nanumfont/raw/master/fonts/NanumFontSetup_TTF_BARUNGOTHIC/NanumBarunGothic.ttf"

# 다운로드 확인
for f in os.listdir(font_dir):
    p = os.path.join(font_dir, f)
    print(f"  {f}  ({os.path.getsize(p):,} bytes)")

# 4) fontManager에 직접 등록 (캐시 재로드 없이)
for f in os.listdir(font_dir):
    if f.endswith(".ttf"):
        try:
            fm.fontManager.addfont(os.path.join(font_dir, f))
        except Exception as e:
            print(f"  addfont 실패 {f}: {e}")

# 5) 등록 확인
installed = sorted({f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name})
print(f"\n등록된 Nanum 폰트: {installed}")

# 6) rcParams 설정
if installed:
    plt.rcParams["font.family"] = installed[0]
    plt.rcParams["axes.unicode_minus"] = False
    print(f"→ 사용 폰트: {installed[0]}")
else:
    print("!! 폰트 등록 실패 — wget 다운로드를 확인하세요")
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
# 폰트 매니저 재로드
fm._load_fontmanager(try_read_cache=False)

# 나눔 계열 폰트 목록
nanum = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
print("설치된 Nanum 폰트:", sorted(set(nanum)))

# 실제 사용할 폰트 선택 (우선순위)
for name in ["NanumGothic", "NanumBarunGothic", "NanumMyeongjo"]:
    if name in nanum:
        plt.rcParams["font.family"] = name
        print(f"→ 사용 폰트: {name}")
        break
else:
    print("!! 나눔폰트 미설치 — apt-get install이 실패했을 수 있음")


폰트 설치 완료
cache dir: /root/.cache/matplotlib
  NanumSquareR.ttf  (723,640 bytes)
  NanumMyeongjoBold.ttf  (4,183,592 bytes)
  NanumBarunGothic.ttf  (0 bytes)
  NanumGothicCoding.ttf  (2,315,924 bytes)
  NanumSquareRoundR.ttf  (1,063,276 bytes)
  NanumGothic.ttf  (0 bytes)
  NanumMyeongjo.ttf  (3,839,464 bytes)
  NanumSquareB.ttf  (733,500 bytes)
  NanumGothicBold.ttf  (4,642,244 bytes)
  NanumBarunGothicBold.ttf  (4,415,956 bytes)
  NanumGothicCodingBold.ttf  (2,246,240 bytes)
  NanumSquareRoundB.ttf  (1,030,948 bytes)
  addfont 실패 NanumBarunGothic.ttf: Can not load face (invalid stream operation; error code 0x55)
  addfont 실패 NanumGothic.ttf: Can not load face (invalid stream operation; error code 0x55)

등록된 Nanum 폰트: ['NanumBarunGothic', 'NanumGothic', 'NanumGothicCoding', 'NanumMyeongjo', 'NanumSquare', 'NanumSquareRound']
→ 사용 폰트: NanumBarunGothic
설치된 Nanum 폰트: ['NanumBarunGothic', 'NanumGothic', 'NanumGothicCoding', 'NanumMyeongjo', 'NanumSquare', 'NanumSquareRound']
→ 사용 폰트: Nanum

In [ ]:
"""구조적 성장 탐지 v3.0 minimal — plot.py (S9)
analysis.py가 만든 CSV만 읽어 4개 그림 + summary.md 생성.
실행: python plot.py [출력폴더=output]
"""
"""구조적 성장 탐지 v3.0 minimal — plot.py (S9) — robust arg parsing
실행: python plot.py [출력폴더=output]
"""
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ---------------------------------------------------------- 폰트
def setup_font():
    for name in ["NanumGothic", "NanumBarunGothic", "Malgun Gothic",
                 "AppleGothic", "Noto Sans CJK KR", "Noto Sans KR"]:
        if any(f.name == name for f in fm.fontManager.ttflist):
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["mathtext.fontset"] = "dejavusans"
    plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25,
                         "grid.linestyle": "--", "figure.dpi": 110,
                         "axes.axisbelow": True})

setup_font()

# ---------------------------------------------------------- 인자 파싱 (robust)
# - 로 시작하는 flag는 무시, 위치 인자만 사용
_positional = [a for a in sys.argv[1:] if not a.startswith("-")]
_candidate_dirs = _positional + ["output", ".", "out"]

OUT = None
for d in _candidate_dirs:
    if os.path.isdir(d) and os.path.exists(os.path.join(d, "series_sa.csv")):
        OUT = d
        break

if OUT is None:
    # series_sa.csv 를 현재 트리에서 검색
    hits = glob.glob("**/series_sa.csv", recursive=True)
    if hits:
        OUT = os.path.dirname(hits[0])
    else:
        print("[ERROR] series_sa.csv 를 찾을 수 없습니다.")
        print("  검색한 후보 폴더:", _candidate_dirs)
        print("  현재 작업 폴더 파일 목록:")
        for f in sorted(os.listdir(".")):
            print("   ", f)
        print("\n해결: analysis.py 를 먼저 실행하거나, "
              "python plot.py <CSV가 있는 폴더> 로 실행하세요.")
        sys.exit(1)

print(f"[plot] OUT = {os.path.abspath(OUT)}")
FIG = os.path.join(OUT, "figures")
os.makedirs(FIG, exist_ok=True)

CODES = [142, 215, 216, 218, 219, 235, 238, 249, 322, 325, 394, 421, 429,
         612, 629, 632, 634, 728]

def load(name):
    p = os.path.join(OUT, name)
    if not os.path.exists(p):
        print(f"  [warn] {name} 없음")
        return pd.DataFrame()
    return pd.read_csv(p)

sa   = load("series_sa.csv")
m1   = load("m1_candidates.csv")
m2   = load("m2_breaks.csv")
eff  = load("m4_effects.csv")
val  = load("m6_validation.csv")
per  = load("m5_persistence.csv")
m7   = load("m7_classification.csv")
ss   = load("m8_shiftshare.csv")
qa   = load("qa_summary.csv")
csum = load("code_summary.csv")

for df_ in (sa, m1, m2, eff, per, m7, ss, qa, csum):
    if not df_.empty and "code" in df_.columns:
        df_["code"] = pd.to_numeric(df_["code"], errors="coerce").astype("Int64")

if not qa.empty:
    inc_mask = qa["included"].astype(str).str.lower().isin(["true", "1"])
    inc_codes = qa.loc[inc_mask, "code"].dropna().astype(int).tolist()
else:
    inc_codes = CODES

print(f"[plot] 포함 코드 {len(inc_codes)}개")
print(f"[plot] sa={len(sa)}, m1={len(m1)}, m2={len(m2)}, eff={len(eff)}, "
      f"val={len(val)}, per={len(per)}, m7={len(m7)}, ss={len(ss)}")

# ============================================================
# (이하 본문은 앞서 드린 plot.py의 fig1~4, summary.md 그대로 사용)
# ============================================================
# ============================================================
# Figure 1 : 코드별 시계열 (18 소패널)
# ============================================================
def fig1_series_panel(sa, m2, codes):
    if sa.empty:
        print("[skip] fig1: series_sa 없음"); return
    sel = m2[m2["status"] == "SELECTED"] if not m2.empty else pd.DataFrame()
    ncol = 3
    nrow = int(np.ceil(len(codes) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 2.8 * nrow), squeeze=False)
    axes = axes.flatten()
    for i, code in enumerate(codes):
        ax = axes[i]
        sub = sa[sa["code"] == code].sort_values("t")
        if sub.empty:
            ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=ax.transAxes)
            ax.set_title(f"{code}", fontsize=10); continue
        t = sub["t"].to_numpy()
        y = sub["logV_sa"].to_numpy()
        ax.plot(t, y, color="#1e88e5", lw=1.5, label="logV_sa")
        if not sel.empty:
            for b in sel[(sel.code == code) & (sel.metric == "V")]["b"].dropna().astype(int):
                ax.axvline(int(b), color="#e53935", ls="--", lw=1.2)
            for b in sel[(sel.code == code) & (sel.metric == "Q")]["b"].dropna().astype(int):
                ax.axvline(int(b), color="#2e7d32", ls=":", lw=1.2)
        ax.set_title(f"{code}", fontsize=10)
        ax.set_xticks(np.arange(0, 72, 12))
        ax.set_xticklabels([str(y) for y in range(2020, 2026)], fontsize=7)
        ax.tick_params(axis="y", labelsize=7)
    for j in range(len(codes), len(axes)):
        axes[j].axis("off")
    handles = [
        Line2D([0], [0], color="#1e88e5", lw=1.5, label="logV_sa (계절조정)"),
        Line2D([0], [0], color="#e53935", ls="--", lw=1.2, label="V break (SELECTED)"),
        Line2D([0], [0], color="#2e7d32", ls=":", lw=1.2, label="Q break (SELECTED)"),
    ]
    fig.legend(handles=handles, loc="upper center", ncol=3, fontsize=9,
               bbox_to_anchor=(0.5, 1.005), frameon=False)
    plt.suptitle("코드별 계절조정 시계열 + SELECTED break",
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG, "fig1_series_panel.png"),
                bbox_inches="tight", dpi=130)
    plt.close()
    print("[fig] fig1_series_panel.png")

fig1_series_panel(sa, m2, inc_codes)


# ============================================================
# Figure 2 : 요약 히트맵 (행=코드, 열=V/Q/UV)
# ============================================================
def fig2_summary_heatmap(m7, codes):
    if m7.empty:
        print("[skip] fig2: m7 없음"); return
    overall_prio = {"STRUCTURAL_GROWTH": 0, "STRUCTURAL_DECLINE": 1,
                    "STRUCTURAL_MIXED": 2, "RECONFIGURATION_ONLY": 3,
                    "CANDIDATE_NOT_SUPPORTED": 4, "NO_CLEAR_CHANGE": 5}
    main_rows = []
    for code in codes:
        sub = m7[m7["code"] == code]
        if sub.empty:
            main_rows.append(dict(code=code, V_status="", Q_status="",
                                  UV_status="", n_breaks=0)); continue
        sub = sub.copy()
        sub["_p"] = sub["overall_status"].map(overall_prio).fillna(99)
        best = sub.loc[sub["_p"].idxmin()]
        main_rows.append(dict(code=code,
                              V_status=best.get("V_status", "") or "",
                              Q_status=best.get("Q_status", "") or "",
                              UV_status=best.get("UV_status", "") or "",
                              n_breaks=len(sub)))
    main = pd.DataFrame(main_rows)

    status_colors = {
        "STRUCTURAL_UP": "#1b5e20",
        "STRUCTURAL_DOWN": "#c62828",
        "STRUCTURAL_MIXED": "#6a1b9a",
        "EFFECT_NOT_SUPPORTED": "#f9a825",
        "SUPPORTED_SMALL_EFFECT": "#90caf9",
        "NO_CLEAR_CHANGE": "#eeeeee",
        "REGRESSION_FAILED": "#424242",
        "": "#f5f5f5",
    }
    short_map = {"STRUCTURAL_UP": "↑", "STRUCTURAL_DOWN": "↓",
                 "STRUCTURAL_MIXED": "M", "EFFECT_NOT_SUPPORTED": "E",
                 "SUPPORTED_SMALL_EFFECT": "s", "NO_CLEAR_CHANGE": "·",
                 "REGRESSION_FAILED": "X", "": "–"}

    fig, ax = plt.subplots(figsize=(6, max(5, 0.32 * len(codes) + 1.5)))
    cols = ["V_status", "Q_status", "UV_status"]
    for i, code in enumerate(codes):
        row = main[main["code"] == code].iloc[0]
        for j, col in enumerate(cols):
            st = row[col]
            ax.add_patch(plt.Rectangle((j, i), 1, 1,
                                       facecolor=status_colors.get(st, "#f5f5f5"),
                                       edgecolor="w", linewidth=0.8))
            txt = short_map.get(st, "?")
            tc = "white" if st in ("STRUCTURAL_UP", "STRUCTURAL_DOWN",
                                    "STRUCTURAL_MIXED", "REGRESSION_FAILED") else "black"
            ax.text(j + 0.5, i + 0.5, txt, ha="center", va="center",
                    fontsize=11, color=tc)
        # n_breaks 표기
        if row["n_breaks"] > 1:
            ax.text(3.05, i + 0.5, f"×{int(row['n_breaks'])}",
                    ha="left", va="center", fontsize=7, color="#616161")

    ax.set_xlim(0, 3); ax.set_ylim(len(codes), 0)
    ax.set_xticks([0.5, 1.5, 2.5])
    ax.set_xticklabels(["Value", "Quantity", "UV"], fontsize=11)
    ax.set_yticks(np.arange(len(codes)) + 0.5)
    ax.set_yticklabels([str(c) for c in codes], fontsize=9)
    ax.set_title("요약 히트맵 — 코드별 가장 강한 break 상태",
                 fontsize=11)
    ax.grid(False)
    handles = [Patch(facecolor=c, edgecolor="w",
                     label=(k if k else "(없음)"))
               for k, c in status_colors.items()]
    ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5),
              fontsize=8, frameon=False)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG, "fig2_summary_heatmap.png"),
                bbox_inches="tight", dpi=130)
    plt.close()
    print("[fig] fig2_summary_heatmap.png")

fig2_summary_heatmap(m7, inc_codes)


# ============================================================
# Figure 3 : Forest plot (V/Q/UV level shift %)
# ============================================================
def fig3_forest_level(eff, val, m2):
    if eff.empty:
        print("[skip] fig3: m4_effects 없음"); return
    eff_ok = eff[eff["status"] == "OK"].copy()
    if eff_ok.empty:
        print("[skip] fig3: 효과 OK 없음"); return

    # level q 병합
    if not val.empty:
        v_l = val[val["effect"] == "level"][["break_id", "metric", "q", "status"]] \
            .rename(columns={"q": "q_level", "status": "val_status"})
        eff_ok = eff_ok.merge(v_l, on=["break_id", "metric"], how="left")
    else:
        eff_ok["q_level"] = np.nan
        eff_ok["val_status"] = "NA"

    order = eff_ok[["break_id", "code", "date"]].drop_duplicates() \
        .sort_values(["code", "date"]).reset_index(drop=True)
    bids = order["break_id"].tolist()
    ymap = {b: i for i, b in enumerate(bids)}

    metrics = ["V", "Q", "UV"]
    colors = {"V": "#1e88e5", "Q": "#2e7d32", "UV": "#fb8c00"}
    fig, axes = plt.subplots(1, 3, figsize=(15, max(4, 0.42 * len(bids) + 2)),
                              sharey=True)
    for k, m in enumerate(metrics):
        ax = axes[k]
        sub = eff_ok[eff_ok["metric"] == m]
        for _, r in sub.iterrows():
            y = ymap.get(r["break_id"])
            if y is None: continue
            sig = (r.get("val_status") == "SUPPORTED")
            alpha = 1.0 if sig else 0.40
            lw = 2.2 if sig else 1.0
            ax.errorbar([r["level_shift_pct"]], [y],
                        xerr=[[max(r["level_shift_pct"] - r["level_ci_lo"], 0)],
                              [max(r["level_ci_hi"] - r["level_shift_pct"], 0)]],
                        fmt="o", color=colors[m], ecolor=colors[m],
                        alpha=alpha, lw=lw, capsize=3, markersize=6,
                        markeredgecolor="k", markeredgewidth=0.4)
        ax.axvline(0, color="k", lw=0.6)
        ax.axvline(10, ls="--", color="#455a64", lw=0.9, alpha=0.7)
        ax.axvline(-10, ls="--", color="#455a64", lw=0.9, alpha=0.7)
        ax.set_xlabel(f"{m} level shift %", fontsize=10)
        ax.set_title(f"지표: {m}", fontsize=11)
        ax.set_ylim(-0.5, len(bids) - 0.5)

    axes[0].set_yticks(range(len(bids)))
    axes[0].set_yticklabels(bids, fontsize=8)
    axes[0].invert_yaxis()

    plt.suptitle("Level shift % (HAC lag2, 95% CI) — 진한 = Level q<0.05 SUPPORTED",
                 fontsize=12, y=1.005)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG, "fig3_forest_level.png"),
                bbox_inches="tight", dpi=130)
    plt.close()
    print("[fig] fig3_forest_level.png")

fig3_forest_level(eff, val, m2)


# ============================================================
# Figure 4 : Shift-Share (2020 vs 2025)
# ============================================================
def fig4_shiftshare(csv_name="m8_shiftshare.csv", out_name="fig4_shiftshare.png",
                    xlabel="시장 대비 초과 성장률 (g_i − g, %p)",
                    title_prefix="Shift-Share (2020 vs 2025)"):
    ss = load(csv_name)
    if ss.empty:
        print(f"[skip] {csv_name} 없음"); return

    # 어떤 컬럼이 있든 excess_pp 계산
    if "excess_pp" in ss.columns:
        pass
    elif "g_i" in ss.columns and "g_mkt" in ss.columns:
        ss["excess_pp"] = (ss["g_i"] - ss["g_mkt"]) * 100
    elif "g_i" in ss.columns and "g" in ss.columns:
        ss["excess_pp"] = (ss["g_i"] - ss["g"]) * 100
    else:
        print(f"[skip] {csv_name}: excess 계산 불가"); return

    ss = ss.dropna(subset=["excess_pp"]).sort_values("excess_pp").reset_index(drop=True)

    CLIP = 100.0
    ss["disp"]    = ss["excess_pp"].clip(-CLIP, CLIP)
    ss["clipped"] = ss["excess_pp"].abs() > CLIP

    fig, ax = plt.subplots(figsize=(11, max(4, 0.42*len(ss)+2)))
    yp = np.arange(len(ss))
    colors = ["#2e7d32" if v >= 0 else "#c62828" for v in ss["excess_pp"]]
    ax.barh(yp, ss["disp"], 0.65, color=colors, edgecolor="white", linewidth=1.2)
    ax.axvline(0, color="k", lw=1.2)
    ax.axvspan(0, CLIP*1.25, alpha=0.06, color="green", zorder=0)
    ax.axvspan(-CLIP*1.25, 0, alpha=0.06, color="red", zorder=0)

    for i, r in ss.iterrows():
        v, d = r["excess_pp"], r["disp"]
        if r["clipped"]:
            ax.text(d + 3, i, f"{v:+.0f}%p ▶", va="center", ha="left",
                    fontsize=9, fontweight="bold", color=colors[i])
        else:
            ax.text(d + (3 if v >= 0 else -3), i, f"{v:+.0f}%p",
                    va="center", ha="left" if v >= 0 else "right",
                    fontsize=9, fontweight="bold", color=colors[i])

    # y 라벨: code 또는 code@YYYY-MM
    labels = []
    for _, r in ss.iterrows():
        if "b" in ss.columns and pd.notna(r.get("b")):
            dt = pd.Period("2020-01","M") + int(r["b"])
            labels.append(f"{int(r['code'])} @{dt.year}-{dt.month:02d}")
        else:
            labels.append(str(int(r["code"])))
    ax.set_yticks(yp); ax.set_yticklabels(labels, fontsize=10, fontweight="bold")

    ax.set_xlim(-CLIP*1.25, CLIP*1.25)
    ax.set_xticks([-100, -50, 0, 50, 100])
    ax.set_xticklabels(["-100", "-50", "0 (시장평균)", "+50", "+100"])
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_title(f"{title_prefix}  ·  n={len(ss)}", fontsize=12, fontweight="bold")

    ax.text( CLIP*0.6, -1.1, "시장 초과 성장", color="#1b5e20",
            fontsize=11, fontweight="bold", ha="center")
    ax.text(-CLIP*0.6, -1.1, "시장 미달 성장", color="#b71c1c",
            fontsize=11, fontweight="bold", ha="center")
    ax.set_ylim(-1.5, len(ss)-0.2)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG, out_name), bbox_inches="tight", dpi=130)
    plt.close()
    print(f"[fig] {out_name}")

# fig4a: S8 (2020 vs 2025)
fig4_shiftshare("m8_shiftshare.csv", "fig4_shiftshare.png",
                xlabel="5년간 초과 성장률 (g_i − g, %p)",
                title_prefix="Shift-Share (2020 vs 2025)")

# fig4b: S8b (break-aligned)
fig4_shiftshare("m8b_break_aligned_shiftshare.csv", "fig4b_shiftshare_break.png",
                xlabel="break 이후 12개월 초과 성장률 (g_i − g_mkt, %p)",
                title_prefix="Shift-Share (break-aligned)")

# ============================================================
# summary.md
# ============================================================
def write_summary():
    lines = ["# 구조적 성장 탐지 v3.0 — 결과 요약", ""]

    # S0
    if not qa.empty:
        n_inc = int(qa["included"].astype(str).str.lower().isin(["true", "1"]).sum())
        n_exc = len(qa) - n_inc
        lines.append(f"## S0 로드·QA")
        lines.append(f"- 포함 코드: {n_inc} / {len(qa)}")
        if n_exc > 0:
            exc = qa[~qa["included"].astype(str).str.lower().isin(["true", "1"])]
            for _, r in exc.iterrows():
                lines.append(f"  - 제외: {int(r['code'])} ({r['reason']})")
        lines.append("")

    # S2
    if not m1.empty:
        det = m1[m1["status"] == "DETECTED"]
        lines.append("## S2 변화점 탐지 (M1)")
        lines.append(f"- V 탐지: {int((det.metric == 'V').sum())}")
        lines.append(f"- Q 탐지: {int((det.metric == 'Q').sum())}")
        lines.append("")

    # S3
    if not m2.empty:
        lines.append("## S3 선정 (M2)")
        vc = m2.groupby(["metric", "status"]).size().unstack(fill_value=0)
        for m in vc.index:
            row = " / ".join(f"{c}={int(vc.loc[m, c])}"
                             for c in vc.columns if c in vc.loc[m].index)
            lines.append(f"- {m}: {row}")
        lines.append("")

    # S4/S5
    if not eff.empty:
        ok = eff[eff["status"] == "OK"]
        lines.append("## S4 효과크기 (M4)")
        lines.append(f"- 회귀 성공: {len(ok)} / {len(eff)}")
        lines.append("")
    if not val.empty:
        lines.append("## S5 통계검증 (M6, BH-FDR)")
        vc = val.groupby(["effect", "status"]).size().unstack(fill_value=0)
        for eff_ in vc.index:
            row = " / ".join(f"{c}={int(vc.loc[eff_, c])}"
                             for c in vc.columns if c in vc.loc[eff_].index)
            lines.append(f"- {eff_}: {row}")
        lines.append("")

    # S6
    if not per.empty:
        lines.append("## S6 지속성 (M5, Level only)")
        vc = per["label"].value_counts()
        for k, v in vc.items():
            lines.append(f"- {k}: {int(v)}")
        lines.append("")

    # S7
    if not m7.empty:
        lines.append("## S7 분류 (M7)")
        vc = m7["overall_status"].value_counts()
        for k, v in vc.items():
            lines.append(f"- {k}: {int(v)}")
        lines.append("")
        lines.append("### break_id 별 (상위 우선순위)")
        prio = {"STRUCTURAL_GROWTH": 0, "STRUCTURAL_DECLINE": 1,
                "STRUCTURAL_MIXED": 2, "RECONFIGURATION_ONLY": 3,
                "CANDIDATE_NOT_SUPPORTED": 4, "NO_CLEAR_CHANGE": 5}
        m7s = m7.copy()
        m7s["_p"] = m7s["overall_status"].map(prio).fillna(99)
        m7s = m7s.sort_values(["_p", "code", "date"]).head(20)
        for _, r in m7s.iterrows():
            lines.append(f"  - {r['break_id']} {int(r['code'])}@{r['date']}  "
                         f"[{r['overall_status']}]  "
                         f"V={r.get('V_status','')}  Q={r.get('Q_status','')}  "
                         f"UV={r.get('UV_status','')}  mech={r.get('mechanism','')}")
        lines.append("")

    # S8
    # ---- S8a: 2020 vs 2025 ----
    if "ss" in globals() and not ss.empty:
        lines.append("## S8a Shift-Share (2020 vs 2025)")
        if "g" in ss.columns and len(ss):
            g = ss["g"].iloc[0]
            lines.append(f"- 시장 성장률 g = {g*100:.2f}%" if np.isfinite(g)
                        else "- g = NA")
        # 초과 성장 상위
        if "excess_pp" not in ss.columns and {"g_i", "g"}.issubset(ss.columns):
            ss["excess_pp"] = (ss["g_i"] - ss["g"]) * 100
        top_hi = ss.dropna(subset=["excess_pp"]).sort_values("excess_pp",
                                                            ascending=False).head(5)
        lines.append("- 시장 초과 성장 상위 5:")
        for _, r in top_hi.iterrows():
            lines.append(f"  - {int(r['code'])}: "
                        f"excess={r['excess_pp']:+.1f}%p  "
                        f"g_i={r['g_i']*100:+.1f}%  "
                        f"actual={r['actual']:,.0f}")
        top_lo = ss.dropna(subset=["excess_pp"]).sort_values("excess_pp").head(3)
        lines.append("- 시장 미달 성장 하위 3:")
        for _, r in top_lo.iterrows():
            lines.append(f"  - {int(r['code'])}: "
                        f"excess={r['excess_pp']:+.1f}%p  "
                        f"g_i={r['g_i']*100:+.1f}%")
        lines.append("")


    # ---- S8b: break-aligned ----
    ssb = load("m8b_break_aligned_shiftshare.csv")
    if not ssb.empty:
        lines.append("## S8b Shift-Share (break-aligned)")
        lines.append(f"- 대상 break 수: {len(ssb)}")
        # g_mkt는 행마다 다름 → 요약 통계만
        if "g_mkt" in ssb.columns:
            lines.append(f"- 시장 성장률(동일기간) 중앙값: "
                        f"{ssb['g_mkt'].median()*100:+.1f}%  "
                        f"(범위 {ssb['g_mkt'].min()*100:+.1f}% ~ "
                        f"{ssb['g_mkt'].max()*100:+.1f}%)")
        if "excess_pp" not in ssb.columns and {"g_i", "g_mkt"}.issubset(ssb.columns):
            ssb["excess_pp"] = (ssb["g_i"] - ssb["g_mkt"]) * 100

        top_hi = ssb.dropna(subset=["excess_pp"]).sort_values(
            "excess_pp", ascending=False).head(5)
        lines.append("- 초과 성장 상위 5 (break 이후 12개월):")
        for _, r in top_hi.iterrows():
            b = int(r["b"]) if "b" in ssb.columns else None
            code_b = f"{int(r['code'])}@{b}" if b is not None else str(int(r['code']))
            lines.append(f"  - {code_b}: excess={r['excess_pp']:+.1f}%p  "
                        f"g_i={r['g_i']*100:+.1f}%  "
                        f"g_mkt={r['g_mkt']*100:+.1f}%")

        top_lo = ssb.dropna(subset=["excess_pp"]).sort_values("excess_pp").head(3)
        if len(top_lo):
            lines.append("- 미달 하위 3:")
            for _, r in top_lo.iterrows():
                b = int(r["b"]) if "b" in ssb.columns else None
                code_b = f"{int(r['code'])}@{b}" if b is not None else str(int(r['code']))
                lines.append(f"  - {code_b}: excess={r['excess_pp']:+.1f}%p  "
                            f"g_i={r['g_i']*100:+.1f}%")
        lines.append("")

        lines.append("## 해석 한계 (보고서 필수)")
        lines.append("- 변화점은 탐색적으로 선택된 뒤 검정되었으므로 p/q는 과대해석 금지 "
                    "(selection after detection).")
        lines.append("- UV는 약가 + 제품 구성 변화가 섞인 대리지표다.")
        lines.append("- Shift-Share의 Differential은 시장(18개군 내부) 대비 상대 성장이며 "
                    "인과·구조적 변화의 증거가 아니다.")
        lines.append("- 국가 단위 집계만으로 정책·기업·처방 원인을 주장하지 않는다.")

        path = os.path.join(OUT, "summary.md")
        with open(path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
        print(f"[md] {path}")

# ============================================================
# summary.md — S8 & S8b 둘 다 처리
# ============================================================
def _fmt_shift_top(df, topn=5, key="differential"):
    """df에서 key 기준 상위 topn 행을 문자열 리스트로."""
    if df.empty: return []
    d = df.dropna(subset=[key]).sort_values(key, ascending=False).head(topn)
    out = []
    for _, r in d.iterrows():
        code = int(r["code"])
        b = int(r["b"]) if "b" in d.columns and pd.notna(r.get("b")) else None
        prefix = f"{code}@{b}" if b is not None else f"{code}"
        out.append(
            f"  - {prefix}: {key}={r[key]:,.0f}  "
            f"g_i={r['g_i']*100:+.1f}%  "
            f"actual={r['actual']:,.0f}")
    return out



write_summary()

print(f"\n[plot] 완료. figures 폴더: {FIG}")

[plot] OUT = /content/output
[plot] 포함 코드 18개
[plot] sa=1296, m1=81, m2=74, eff=150, val=300, per=150, m7=50, ss=18
[fig] fig1_series_panel.png
[fig] fig2_summary_heatmap.png
[fig] fig3_forest_level.png


/tmp/ipykernel_3015/646549937.py:365: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_3015/646549937.py:366: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) NanumGothic.
  plt.savefig(os.path.join(FIG, out_name), bbox_inches="tight", dpi=130)


[fig] fig4_shiftshare.png
  [warn] m8b_break_aligned_shiftshare.csv 없음
[skip] m8b_break_aligned_shiftshare.csv 없음
  [warn] m8b_break_aligned_shiftshare.csv 없음

[plot] 완료. figures 폴더: output/figures


# 보고서 문장 초안

v3.0 파이프라인(단일 config PELT + ±12M ITS + BH-FDR)은 18개 약효분류군에서 총 50개 구조 변화점을 탐지했다. 이 중 STRUCTURAL_GROWTH 17건, STRUCTURAL_DECLINE 10건, RECONFIGURATION_ONLY 12건으로 분류되었다.

가장 강한 성장 신호는 **249@2024-01, 728@2023-03, 238@2025-02**로, 세 지표(V/Q/UV)가 모두 STRUCTURAL_UP이며 지속성 라벨이 AMPLIFIED/PERSISTENT다. **429@2024-01**은 V와 UV가 급증했으나 Q는 통계적으로 지지되지 않아 단가/제품구성 변화(UNIT_VALUE_DRIVEN)로 분류된다. 다만 동일 코드의 **2024-11** break에서 STRUCTURAL_DOWN이 탐지되어 최근 반전 조짐이 관찰된다.

**728**은 단일 코드 내에서 7개 break가 탐지되어, 2020~2021 쇠퇴 → 2022~2024 성장 → 2024-09 재편의 다중 regime 이력을 보였다. **612와 629**는 2023~2024에 RECONFIGURATION_ONLY 또는 MIXED 국면이 다수로, 단순 성장이 아닌 가격/구성 재편이 진행 중인 것으로 해석된다.

# 주의 문구

v3.0은 단일 config 결과이므로 support 미검증 break가 다수 포함됨

**429**의 이중 국면(B026 UP, B027 DOWN)은 시계열 직접 확인 필요

UV는 실제 약가가 아니므로 UNIT_VALUE_DRIVEN은 "가격+구성 혼재"로만 서술